# Create catalog of sources to create cutouts for
Catalog contains L4 detected source positions, filter image file locations (on AWS), Kron AB magnitude, and detection segment area

In [1]:
import s3fs #To access cloud files as though they were local
from pprint import pprint
import asdf #To handle ASDF input and output
import pyarrow.parquet as pq

import numpy as np

import pandas as pd

from datetime import datetime

from tqdm import trange

In [2]:
def Retrieve_file_matches(dir_uri, filter_match, All_files=True, in_plan=None, in_pass=None, in_segment=None):
    '''
    
    Retrieves a list of the files stored on AWS that match 
    the input dir_uri root directory, and input filter, plan, 
    pass, and segment. Also splits off separate arrays that 
    contain the visit-level info and sky-cell info associated 
    with each returned file. If All_files = True, returns all 
    files that match the input filter. Otherwise, an input 
    plan, pass, and segment 
    are also required.

    Parameters:
    -----------
    dir_uri: str
        the input directory to parse
    filter_match: str
        the filter to match against
    All_files: boolean
        Whether to return all files that match the input filter
        or impose filter on plan, pass, and segment as well
    in_plan: str
        The plan number. A specification in APT of the Pass Plans  
        comprising a Program. Should have format "01" for plan 1
    in_pass: str
        The pass number. The collection of activities generated from 
        each iteration of a Pass Plan. Multiple Passes may be generated 
        from the same Pass Plan to allow repetition or execute at 
        different orientations (via special requirements).
    in_segment: str
        The sequence of activities produced by each iteration of the 
        Observation Sequences defined in a Segment Plan. A Segment 
        may include multiple traversals of Mosaic Pattern(s), with 
        filter wheel moves occurring between Observations. The 
        Segment will be the unit of planning, while short term 
        scheduling will be done at the Visit level.
    
    Returns:
    --------
    visit_level_info_array: nd.array
        array of the viti-level combo information for each matched 
        program file.
    skycell_info_array: nd.array
        the skycell information for each matched program file
    prog_match_files: list
        the list of program files matched to the input filter, plan, 
        pass, and segment 
    '''
    file_list = fs.ls(dir_uri)
    print(f'Full length of file list available at {dir_uri} = ', len(file_list))
    
    print(f'Looking for files taken in filter {filter_match}')
    
    prog_match_files = []
    
    visit_level_info = []
    skycell_info = []
    filter_info = []
    suffix = []
    for i in range(len(file_list)):
    # for i in range(5): #v01001001002001
        suffix_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-1].rsplit('.')[0]
        visitlev_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-4]
        filter_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-2]
        skycell_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-3]
        # print('visit-level-info = ', visitlev_inf)
        plan_current = visitlev_inf[1:3]
        pass_current = visitlev_inf[3:6]
        segment_current = visitlev_inf[6:9]
        # print(plan_current, pass_current, segment_current)
        
        
        # filter_info.append(filter_inf)
        if All_files == True:
            if filter_inf == filter_match and suffix_inf == 'coadd':

                visit_level_info.append(visitlev_inf)
                skycell_info.append(skycell_inf)
                prog_match_files.append(file_list[i])
        else:
            if filter_inf == filter_match and suffix_inf == 'coadd' and plan_current== in_plan and pass_current == in_pass and segment_current == in_segment:

                visit_level_info.append(visitlev_inf)
                skycell_info.append(skycell_inf)
                prog_match_files.append(file_list[i])

    print(f'Summary of files:')        
    if len(visit_level_info) > 0:
        len_fl = len(visit_level_info)
        print(f'{len_fl} files found that match input filter request!')
        # pprint(visit_level_info)
    else:
        print('No files found!')

    visit_level_info_array = np.array(visit_level_info)
    skycell_info_array = np.array(skycell_info)
        
    return visit_level_info_array, skycell_info_array, prog_match_files

def Print_file_info(visit_level_info_array, skycell_info_array):
    '''
    Given an input array of visit-level combinations and skycells, 
    print a summary of the visit-level and skycell information.  

    Prameters:
    ---------
    visit_level_info_array: nd.array
        an array of visit-level descriptors that summarize the 
        plan, pass, segment, observations and visit of each file
    
    skycell_info_array: nd.array
        an array of the skycell info associated with each 
        visit-level descriptor of the input program/filter

    Returns:
    --------
    None
        
    '''

    print('Number of unique visit-level combos :', len(np.unique(visit_level_info_array)))
    print('Number of unique skycell positions:', len(np.unique(skycell_info_array)))
    unique_visit_level_info = np.unique(visit_level_info_array)
    

    PASS_info = []
    SEG_info = []
    OBS_info = []
    VISIT_info = []
    SKYCELL_info = []
    
    
    for i in range(len(unique_visit_level_info)):
        VL_info = unique_visit_level_info[i]
        # these are hard coded to match the format of visit-level info of Roman WFI files
        # https://roman-docs.stsci.edu/data-handbook/data-levels-and-products
        pass_info = VL_info[3:6]
        seg_info = VL_info[6:9]
        obs_info = VL_info[9:12]
        visit_info = VL_info[12:]

        PASS_info.append(pass_info)
        SEG_info.append(seg_info)
        OBS_info.append(obs_info)
        VISIT_info.append(visit_info)
    
        visit_match = np.where(visit_level_info_array==VL_info)[0]
        skycell_match = skycell_info_array[visit_match]

        
    unique_PASS_info = np.unique(PASS_info)
    num_passes = len(unique_PASS_info)
    print(f'Found {num_passes} unique passes: ', unique_PASS_info)
    
    unique_SEF_info = np.unique(SEG_info)
    num_segs = len(unique_SEF_info)
    print(f'Found {num_segs} unique segments', unique_SEF_info)
    
    unique_OBS_info = np.unique(OBS_info)
    num_obs = len(unique_OBS_info)
    print(f'Found {num_obs} unique observations', unique_OBS_info)
    
    PASS_info_array = np.array(PASS_info)
    SEG_info_array = np.array(SEG_info)
    OBS_info_array = np.array(OBS_info)
    VISIT_info_array = np.array(VISIT_info)
    
    print('======================================')
    for pid in unique_PASS_info:

        id_match = np.where(PASS_info_array == pid)
        seg_match = SEG_info_array[id_match]
        obs_match = OBS_info_array[id_match]
        visit_match = VISIT_info_array[id_match]
    
        print(f'Pass {pid} has...')
        print(f'{len(np.unique(seg_match))} segment: {np.unique(seg_match)}')
        print(f'{len(np.unique(obs_match))} observations: {np.unique(obs_match)}')
        print(f'{len(np.unique(visit_match))} visits: {np.unique(visit_match)}')
        print('======================================')

### Check the available simulation images in the AWS S3 bucket for the SOC simulations:
<!--  -->
- an example of the file naming structure: "r00342_p_v0100100100100_008m46x51y68_f087"

- the first part, "r00342," is the program

- "p" indicates that this is a prompt data product (not an officiel data release project)

- each file has a visit-level descriptor: "v01002001001001," which can be broken into components "v01|002|001|001|001"
    - This is a visit-level combination of co-added (dithered) exposures from execution plan 01, pass 002, segment 001, observation 001, visit 001. 

- pass 013 corresponds to the medium tier HLWAS (https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts)


- for each filter, this pass (013) should have plan 01, segment 001, and visits 001-018 (one for each tile of the HLWAS mosaic)


- for each filter, the "observation" descriptor varies: f106: 001; f129: 002; f158: 003


- Need to indentify all files that correspond to plan 01, pass 013, and segment 001


- the second-to-last part of the file naming structure is the skycell. The skycell information "008m46x51y68" contains the center of the projection region (008m46 = RA,DEC = 08, -46) and the center of the sky cell within the proection region in pixel coords (x51y68 = x,y = 51,68)


In [3]:
'''
find out how many unique visit-level combinations for each filter to help get an idea of what "observations" this dataset includes
'''
# file_source = 'openuniverse2024'
file_source = 'soc_simulations'

# the root directory of the program
dir_uri = 's3://stpubdata/roman/nexus/soc_simulations/r00342/'
# list all of the available simulated dataproducts
# we need filters YJH
in_filter = 'f106' #primary filter
f1 = 'f129' #secondary filter
f2 = 'f158' #tertiary filter
bands = [in_filter, f1, f2]
src_mag_limit=20.5 #magntiude limit is somewhat arbitrary, was initially chosen to match the magntiude limit of mos galaxies in the GZ: Hubble project (23.5), but changed to limit the number of objects identified for testing purposes

in_plan_ = '01'
in_pass_ = '013'
in_segment_ = '001'
# in_observation = '002'
# in_visit = '001'

fs = s3fs.S3FileSystem(anon=True) #connects us to AWS S3 bucket where the data is stored; anon=True means we only want access to public data files

band_info = {}
if file_source == 'soc_simulations':
    for i in range(len(bands)):
        print(f'=========================================={bands[i]}==========================================')
        filter_visit_level_info_array, filter_skycell_info_array, filter_prog_match_files = Retrieve_file_matches(dir_uri+'l3/', bands[i], All_files=False, in_plan=in_plan_, in_pass=in_pass_, in_segment=in_segment_)
        Print_file_info(filter_visit_level_info_array, filter_skycell_info_array)
        band_info[bands[i]] = {'visit_level_info': filter_visit_level_info_array, 
                          'skycell_info_array':filter_skycell_info_array, 
                          'prog_match_files':filter_prog_match_files}

==========================================f106==========================================
Full length of file list available at s3://stpubdata/roman/nexus/soc_simulations/r00342/l3/ =  554352
Looking for files taken in filter f106
Summary of files:
2670 files found that match input filter request!
Number of unique visit-level combos : 18
Number of unique skycell positions: 1450
Found 1 unique passes:  ['013']
Found 1 unique segments ['001']
Found 1 unique observations ['001']
Pass 013 has...
1 segment: ['001']
1 observations: ['001']
18 visits: ['001' '002' '003' '004' '005' '006' '007' '008' '009' '010' '011' '012'
 '013' '014' '015' '016' '017' '018']
==========================================f129==========================================
Full length of file list available at s3://stpubdata/roman/nexus/soc_simulations/r00342/l3/ =  554352
Looking for files taken in filter f129
Summary of files:
2662 files found that match input filter request!
Number of unique visit-level combos : 18


# Start from the visit-level combinations returned for the f106 filter.
- for every program file that matches the in_filter and pass_number (which is a observation + visit + skycell combination), identify the file name for the match_filters that overlap with the skycell
- the matched files in each filter may contain up to 4 visits (single pointings) that I'm guessing overlap at the given skycell. The visit numbers should be adjacent to each other in the mosaic (e.g., 1,2,11,12 or 2,3,10,11 becuase simulated moasic proceeds in an S-pattern - see https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts )
- ##### what is the best way to select the "best" visit for a given skycell (for skycells where multiple pointings (visits) overlap??)
    - could take the first one in the returned list but it may not cover the galaxy completely and we'd have to figure out which one is the best?
    - could create all returned images, but then still need to decide how to deduplicate later on...
    - also, some visits may have more coverage of the target than others...how to figure out which visit has the most coverage of the target?

In [4]:
# organize all of the visit-level and sky-cell info for each non-primary filter
# should have the following format:
# match_filters_info = {f1:{'visit_level_info': f1_visit_level_info_array, 
#                           'skycell_info_array':f1_skycell_info_array, 
#                           'prog_match_files':f1_prog_match_files}, 
#                       f2:{'visit_level_info': f2_visit_level_info_array, 
#                           'skycell_info_array':f2_skycell_info_array, 
#                           'prog_match_files':f2_prog_match_files}}
#                       ...
#                       fn:{'visit_level_info': fn_visit_level_info_array, 
#                           'skycell_info_array':fn_skycell_info_array, 
#                           'prog_match_files':fn_prog_match_files}}

match_filters = bands[1:]
match_filters_info = {}
for i in range(len(match_filters)):
    match_filters_info[match_filters[i]] = band_info[match_filters[i]]


In [5]:
# create empty source catalog

catalog_columns = ['id_str']
for i in range(len(bands)):
    catalog_columns.append(f'asdf_loc_{bands[i]}')

catalog_columns.extend(['RA', 'DEC', f'KronABmag_{bands[0]}', f'kron_radius_{bands[0]}', f'kron_radius_flux50_{bands[0]}', f'segment_area_{bands[0]}'])
print(catalog_columns)

catalog_df = pd.DataFrame(columns=catalog_columns)

print(catalog_df)

['id_str', 'asdf_loc_f106', 'asdf_loc_f129', 'asdf_loc_f158', 'RA', 'DEC', 'KronABmag_f106', 'kron_radius_f106', 'kron_radius_flux50_f106', 'segment_area_f106']
Empty DataFrame
Columns: [id_str, asdf_loc_f106, asdf_loc_f129, asdf_loc_f158, RA, DEC, KronABmag_f106, kron_radius_f106, kron_radius_flux50_f106, segment_area_f106]
Index: []


In [6]:
'''
pipeline catalog parameters:
['label', 'flagged_spatial_id', 'x_centroid', 'y_centroid',
       'x_centroid_err', 'y_centroid_err', 'x_centroid_win', 'y_centroid_win',
       'x_centroid_win_err', 'y_centroid_win_err', 'x_psf', 'x_psf_err',
       'y_psf', 'y_psf_err', 'ra', 'dec', 'ra_centroid', 'dec_centroid',
       'ra_centroid_err', 'dec_centroid_err', 'ra_centroid_win',
       'dec_centroid_win', 'ra_centroid_win_err', 'dec_centroid_win_err',
       'ra_psf', 'dec_psf', 'ra_psf_err', 'dec_psf_err', 'bbox_xmin',
       'bbox_xmax', 'bbox_ymin', 'bbox_ymax', 'segment_area', 'semimajor',
       'semiminor', 'fwhm', 'ellipticity', 'orientation_pix',
       'orientation_sky', 'cxx', 'cxy', 'cyy', 'kron_radius', 'nn_label',
       'nn_distance', 'sharpness', 'roundness1', 'is_extended',
       'fluxfrac_radius_50', 'aper_bkg_flux', 'aper_bkg_flux_err',
       'aper01_flux', 'aper01_flux_err', 'aper02_flux', 'aper02_flux_err',
       'aper04_flux', 'aper04_flux_err', 'aper08_flux', 'aper08_flux_err',
       'aper16_flux', 'aper16_flux_err', 'psf_flux', 'psf_flux_err',
       'segment_flux', 'segment_flux_err', 'kron_flux', 'kron_flux_err',
       'kron_abmag', 'kron_abmag_err', 'warning_flags', 'image_flags',
       'psf_flags', 'psf_gof']
'''
new_row_empty = {}
for i in range(len(catalog_columns)):
    new_row_empty[catalog_columns[i]] = None
print(new_row_empty)

{'id_str': None, 'asdf_loc_f106': None, 'asdf_loc_f129': None, 'asdf_loc_f158': None, 'RA': None, 'DEC': None, 'KronABmag_f106': None, 'kron_radius_f106': None, 'kron_radius_flux50_f106': None, 'segment_area_f106': None}


In [7]:
# initialize the empty MATCH_FILTER_IMAGES dictionary
# this will be filled with the file information for each 
# filter that matches the input filter skycell (so it is 
# reset to empty for each iteration through primary filter's prog_match_files)
MATCH_FILTER_IMAGES_empty = {}
for i in range(len(match_filters)):
    MATCH_FILTER_IMAGES_empty[match_filters[i]] = {'prog_match_files':None}
print(MATCH_FILTER_IMAGES_empty)

{'f129': {'prog_match_files': None}, 'f158': {'prog_match_files': None}}


In [8]:
# FWHM (miliarcsec) for f106, f129, and f158 = 87, 106, 128 https://roman-docs.stsci.edu/roman-instruments/the-wide-field-instrument/observing-with-the-wfi/wfi-quick-reference#WFIQuickReference-Table-WFI-Filters
FWHM = np.mean([87, 106, 128])/(1e3)
PSF_area = 1.133*pow(FWHM,2)
print(PSF_area, 'arcsec^2')
src_id = 0

0.012971716999999999 arcsec^2


In [9]:

startTime = datetime.now()
for i in trange(len(band_info[bands[0]]['prog_match_files'])):
# for i in range(3): 
    band_files = []
    # =============================================================================================
    # define primary filter image and catalog URIs for the current visit- and skycell-level descriptor
    file_name_coadd = 's3://'+band_info[bands[0]]['prog_match_files'][i]
    file_name_cat = dir_uri+'l4/l3_source_catalog/'+file_name_coadd.rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'

    # print(f'INPUT FILE NAME: {file_name_coadd}')
    # print(f'USING L4 CATALOG {file_name_cat} \n')
    band_files.append(file_name_coadd)

    # =============================================================================================
    # create empty dictionary to store the full URIs that cover the same visit- and skycell-level descriptor as the primary filter 
    MATCH_FILTER_IMAGES = MATCH_FILTER_IMAGES_empty
    # print('len(MATCH_FILTER_IMAGES) = ', len(MATCH_FILTER_IMAGES))
    
    # for the non-primary filters, identify where their skycell info matches the current primary filter skycell info
    for j in range(len(match_filters)):
        f = match_filters[j]
        match_i = np.where(match_filters_info[f]['skycell_info_array']==band_info[bands[0]]['skycell_info_array'][i])[0]
        
        # print(f'    The following images were indentified for filter {f}:')
        # pprint(np.array(match_filters_info[f]['prog_match_files'])[match_i])

        len_match = len(np.array(match_filters_info[f]['prog_match_files'])[match_i])
        if len_match>0:
            MATCH_FILTER_IMAGES[f]['prog_match_files'] = np.array(match_filters_info[f]['prog_match_files'])[match_i]
    
        # report how many pointings (visits) of the secondary or tertiary filter overlap with the current visit- and skycell-level descriptor of the primary filter 
        # print(f"    {len_match} pointings in filter {f} match the input skycell \n")
        
        # print(stop)
        # Once at the last filter, check that files were found for all match filters
        missing_filter = False
        if f == match_filters[-1]:
        # and isinstance(MATCH_FILTER_IMAGES[f]['prog_match_files'], np.ndarray) and isinstance(MATCH_FILTER_IMAGES[match_filters[i-1]]['prog_match_files'], np.ndarray):
            
            for k in range(len(MATCH_FILTER_IMAGES)):
                
                if isinstance(MATCH_FILTER_IMAGES[match_filters[k]]['prog_match_files'], np.ndarray):
                    continue
                else:
                    missing_filter = True

            if missing_filter == False:
                # print('    Found skycell file matches for all filters! \n')
                # print('\n')
                # print('    Taking first pointing of input filter match list \n')
                for m in range(len(MATCH_FILTER_IMAGES)):
                    band_files.append('s3://'+MATCH_FILTER_IMAGES[match_filters[m]]['prog_match_files'][0])
                # print('band_files = ', band_files)
                
                # # open the image and extract data array and metadata
                # data_, allwcs_, pixel_scale_ = open_image_asdf(file_name_coadd)
                # wcs_ = allwcs_[0]
                # gwcs_ = allwcs_[1]
            
                # #  ===================================================================================
                # read in associated catalog file:
                # Read the file metadata schema
                parquet_file = pq.read_table(file_name_cat, filesystem=fs)
                columns = parquet_file.schema.names
                cat_table = parquet_file.to_pandas()
                
                # extract RA,DEC positions for all sources in the current catalog
                srcRA_, srcDEC_, kron_abmag, kron_rad, kron_rad_flux50, segment_area = np.array(cat_table['ra_centroid'].values), np.array(cat_table['dec_centroid'].values), np.array(cat_table['kron_abmag'].values), np.array(cat_table['kron_radius'].values), np.array(cat_table['fluxfrac_radius_50'].values), np.array(cat_table['segment_area'].values)
    
                # identify which catalog sources are extended and have kron_abmag < 23.5
                ext_i = np.where((cat_table['is_extended']) & (kron_abmag<src_mag_limit))[0] 
    
                # extract RA,DEC positions for sources in the current catalog that meet the selection criteria
                srcRA_, srcDEC_, kron_abmag, segment_area = srcRA_[ext_i], srcDEC_[ext_i], kron_abmag[ext_i], segment_area[ext_i]
                num_src_in_image = len(ext_i)
                # print(f'number of extended sources in this skycell w/ Kron AB magnitude < {src_mag_limit} = {num_src_in_image} \n')
                if num_src_in_image == 0:
                    continue
                # print('Displaying image with identified sources overlaid')
                # plt.figure(figsize=(10, 8))
                # ax2 = plt.subplot(projection=wcs_)
                # plt.imshow(data_, vmin=0, vmax=5, origin='lower')
                # # plt.scatter(pixels[0], pixels[1], marker='*', color='red')
                # ax2.scatter(srcRA_, srcDEC_, marker='o', facecolor='none', edgecolor='red', transform=ax2.get_transform('world'))
                # plt.show()
    
                # add each source selected from current skycell to the catalog
                for l in range(len(srcRA_)):
                    
                    ra_, dec_ = srcRA_[l], srcDEC_[l]
    
                    # im_cut, scales_ = create_image_cutout_test(band_files, cutout_size_, [srcRA_[l], srcDEC_[l]], display_full=False, full_data = [data_,wcs_])
                    
                    # cutouts_ = im_cut
                    
                    # {'id_str': None, 'asdf_loc_f106': None, 'asdf_loc_f129': None, 'asdf_loc_f158': None, 'RA': None, 'DEC': None, 'KronABmag_f106': None}
                    new_row = new_row_empty
                    new_row['id_str'] = src_id
                    for b in range(len(band_files)):
                        new_row[f'asdf_loc_{bands[b]}'] = band_files[b]
                    new_row['RA'] = ra_
                    new_row['DEC'] = dec_
                    new_row[f'KronABmag_{bands[0]}'] = kron_abmag[l]
                    new_row[f'kron_radius_{bands[0]}'] = kron_rad[l]
                    new_row[f'kron_radius_flux50_{bands[0]}'] = kron_rad_flux50[l]
                    new_row[f'segment_area_{bands[0]}']= segment_area[l]
                    # print('new_row = ', new_row)
                    catalog_df = pd.concat([catalog_df, pd.DataFrame([new_row])], ignore_index=True)
                    src_id+=1
            # else:
                # print('Missing a filter for this skycell! skipping!')
    # print('==============================================================================')

duration = datetime.now()-startTime

100%|██████████| 2670/2670 [15:27<00:00,  2.88it/s]


In [ ]:
print('Time to create catalog:', duration)
print(catalog_df)
catalog_name = 'catalog_file_locs_'
for i in range(len(bands)):
    if bands[i] == bands[-1]:
        catalog_name = catalog_name + bands[i]
    else:
        catalog_name = catalog_name + bands[i]+'_'
print(catalog_name)
catalog_df.to_parquet('/teams/gz-roman/'+catalog_name+'.parquet')

In [ ]:
cat_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')
# make sure there is only one entry per source id. At one point, the 
# above loop reset the src_id counter so this is mostly to test that 
# doesn't happen again. Eventually, we should remove this cell once 
# we are sure that issue does not pop up again.
print(len(cat_df))
print(len(np.unique(cat_df['id_str'])))
# print(np.unique()

# Get the indices of the first occurrences
_, first_occurrence_indices = np.unique(cat_df['id_str'], return_index=True)

# Create a boolean mask of all False values
mask = np.zeros_like(cat_df['id_str'], dtype=bool)

# Mark the first occurrences as True
mask[first_occurrence_indices] = True

# Invert the mask to find all repeated/duplicate indices
repeated_indices = np.where(~mask)[0]

print("Repeated indices:", repeated_indices)
print(len(repeated_indices))
print(len(np.unique(repeated_indices)))